In [1]:
# ============================================================
# Gated Fusion: CLIP + ViLT — FakeNewsNet 2-Way — 5,000 Samples
# Kaggle Notebook | 5-Fold CV | Oversample → 5000 | 10 Epochs
# ============================================================

import os, random, warnings, json, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    CLIPProcessor, CLIPModel,
    ViltProcessor, ViltModel,
    get_linear_schedule_with_warmup,
)
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report,
)
from tqdm import tqdm
warnings.filterwarnings("ignore")

try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False

# ── Reproducibility & Device ──────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
seed_everything()

DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES      = 2
NUM_EPOCHS       = 10
BATCH_SIZE       = 32
LR               = 2e-4
N_FOLDS          = 5
MAX_SAMPLES      = 5000          # total after oversampling (2500 per class)
FREEZE_BACKBONES = True
EARLY_STOP_PAT   = 3
VAL_SPLIT        = 0.10          # fraction of train+val held out for val per fold
CLASS_NAMES      = ["real", "fake"]
CKPT_DIR         = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def make_autocast():
    try:    return torch.amp.autocast("cuda")
    except: return torch.cuda.amp.autocast()

def make_scaler():
    try:    return torch.amp.GradScaler("cuda")
    except: return torch.cuda.amp.GradScaler()

USE_AMP = DEVICE.type == "cuda"

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── STEP 1: Discover every file under /kaggle/input ───────
print("\n=== Scanning /kaggle/input ===")
all_files = []
all_dirs  = []
for root, dirs, files in os.walk("/kaggle/input"):
    # skip hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    all_dirs.append(root)
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"Total files : {len(all_files)}")
print(f"Total dirs  : {len(all_dirs)}")

# Show top-level datasets attached
top_level = [d for d in os.listdir("/kaggle/input")
             if os.path.isdir(f"/kaggle/input/{d}")]
print(f"Attached datasets: {top_level}")

# Show first 60 files so we can see the structure
print("\nFirst 60 files found:")
for f in sorted(all_files)[:60]:
    print(f"  {f}")

# ── STEP 2: Detect what kind of dataset is present ────────
def detect_dataset_type():
    """
    Returns one of:
      'fakenewsnet_json'   — classic JSON structure (gossipcop/fake/<id>/news content.json)
      'fakenewsnet_csv'    — CSV-based (gossipcop.csv / politifact.csv / fakenewsnet.csv)
      'generic_csv'        — any CSV with title + label columns
      None                 — nothing usable found
    """
    # Check for JSON structure
    json_hits = [f for f in all_files if f.endswith("news content.json")]
    if json_hits:
        print(f"\nDetected: FakeNewsNet JSON structure ({len(json_hits)} articles)")
        return "fakenewsnet_json", json_hits

    # Check for known CSV names — only actual .csv files
    csv_names = {"gossipcop.csv", "politifact.csv", "fakenewsnet.csv",
                 "fake_news.csv", "news.csv", "data.csv", "train.csv",
                 "gossipcop_v3.csv", "politifact_v3.csv"}
    csv_hits = [f for f in all_files
                if f.lower().endswith(".csv") and (
                    os.path.basename(f).lower() in csv_names
                    or "fakenews" in f.lower() or "gossipcop" in f.lower()
                    or "politifact" in f.lower() or "_fake_" in f.lower()
                    or "_real_" in f.lower()
                )]
    if csv_hits:
        print(f"\nDetected: CSV-based dataset")
        for c in csv_hits: print(f"  {c}")
        return "fakenewsnet_csv", csv_hits

    # Fallback: any CSV with >=100 rows
    any_csvs = [f for f in all_files if f.endswith(".csv")]
    if any_csvs:
        print(f"\nDetected: Generic CSVs ({len(any_csvs)} files)")
        for c in any_csvs[:10]: print(f"  {c}")
        return "generic_csv", any_csvs

    return None, []

dataset_type, dataset_files = detect_dataset_type()
print(f"\nDataset type: {dataset_type}")

# ── STEP 3: Loaders for each detected type ────────────────
_IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

def _find_first_image(article_dir):
    img_folder = os.path.join(article_dir, "images")
    if not os.path.isdir(img_folder):
        return None
    for fname in sorted(os.listdir(img_folder)):
        if os.path.splitext(fname)[1].lower() in _IMG_EXTS:
            fpath = os.path.join(img_folder, fname)
            if os.path.getsize(fpath) > 500:
                return fpath
    return None

def load_from_json(json_files):
    """Load from news content.json structure."""
    records = []
    for jf in tqdm(json_files, desc="Loading JSON articles"):
        # path: .../<source>/<label>/<id>/news content.json
        parts = jf.replace("\\", "/").split("/")
        try:
            label_str  = parts[-3].lower()   # fake / real
            source_str = parts[-4].lower()   # gossipcop / politifact
        except IndexError:
            continue
        if label_str not in ("fake", "real"):
            continue
        label_int = 1 if label_str == "fake" else 0
        art_dir   = os.path.dirname(jf)
        try:
            with open(jf, "r", encoding="utf-8", errors="ignore") as fh:
                data = json.load(fh)
            title = str(data.get("title", "")).strip()
            body  = str(data.get("text",  "")).strip()[:256]
            text  = (title + " " + body).strip() if body else title
            if not text:
                continue
        except Exception:
            continue
        records.append({
            "text":     text,
            "img_path": _find_first_image(art_dir),
            "label":    label_int,
            "source":   source_str,
        })
    return pd.DataFrame(records)

def _try_read_csv(path):
    """Try reading a CSV/TSV with multiple encodings."""
    for sep in [",", "\t"]:
        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc,
                                 low_memory=False, on_bad_lines="skip")
                if len(df) > 10:
                    return df
            except Exception:
                continue
    return None

def load_from_csv(csv_files):
    """
    Load from CSV files.
    Handles two cases:
      1. Files with an explicit label column (label/fake/real/class/…)
      2. FakeNewsNet-style split files where the label is in the filename
         (e.g. BuzzFeed_fake_news_content.csv → label=1,
               BuzzFeed_real_news_content.csv → label=0)
    """
    TEXT_COLS  = ["title", "clean_title", "text", "headline", "content", "body",
                  "news_title", "statement"]
    LABEL_COLS = ["label", "fake", "real", "is_fake", "class", "target",
                  "category", "veracity", "fake_or_real"]

    def _label_from_filename(path):
        """Return 1/0 if filename clearly signals fake/real, else None."""
        name = os.path.basename(path).lower()
        if "_fake_" in name or name.startswith("fake"):
            return 1
        if "_real_" in name or name.startswith("real"):
            return 0
        return None

    all_records = []
    for csv_path in csv_files:
        df_raw = _try_read_csv(csv_path)
        if df_raw is None or len(df_raw) < 5:
            continue
        cols_lower = {c.lower(): c for c in df_raw.columns}

        # find text column
        text_col = next((cols_lower[c] for c in TEXT_COLS if c in cols_lower), None)
        if text_col is None:
            text_col = next(
                (c for c in df_raw.columns
                 if "title" in c.lower() or "text" in c.lower()), None)
        if text_col is None:
            continue

        # try to find label column in the data
        label_col = next((cols_lower[c] for c in LABEL_COLS if c in cols_lower), None)
        if label_col is None:
            label_col = next(
                (c for c in df_raw.columns
                 if "label" in c.lower() or "fake" in c.lower()
                 or "real" in c.lower() or "class" in c.lower()), None)

        # fall back: infer label from filename
        filename_label = _label_from_filename(csv_path)
        if label_col is None and filename_label is None:
            print(f"  SKIP {os.path.basename(csv_path)}: no label column and label not in filename")
            continue

        print(f"  CSV {os.path.basename(csv_path)}: "
              f"text='{text_col}' "
              f"label={'(from filename='+str(filename_label)+')' if label_col is None else repr(label_col)}"
              f" rows={len(df_raw)}")

        for _, row in df_raw.iterrows():
            text = str(row.get(text_col, "")).strip()
            if len(text) < 5:
                continue

            if label_col is not None:
                raw_label = str(row.get(label_col, "")).strip().lower()
                if raw_label in ("1", "fake", "false", "mostly false", "pants on fire"):
                    lbl = 1
                elif raw_label in ("0", "real", "true", "mostly true", "half true"):
                    lbl = 0
                else:
                    try:
                        lbl = int(float(raw_label))
                        if lbl not in (0, 1):
                            continue
                    except Exception:
                        continue
            else:
                lbl = filename_label  # already validated above

            all_records.append({"text": text[:512], "img_path": None,
                                 "label": lbl, "source": "csv"})

    return pd.DataFrame(all_records) if all_records else pd.DataFrame()

# ── STEP 4: Load data ─────────────────────────────────────
# Cap files before loading to avoid processing far more records than needed.
# We pre-select ~3× MAX_SAMPLES to ensure enough survive any filtering.
_LOAD_CAP = MAX_SAMPLES * 3
if dataset_type == "fakenewsnet_json" and len(dataset_files) > _LOAD_CAP:
    _fake_j = [f for f in dataset_files if "/fake/" in f.replace("\\", "/")]
    _real_j = [f for f in dataset_files if "/real/" in f.replace("\\", "/")]
    _n_each = _LOAD_CAP // 2
    dataset_files = (
        random.sample(_fake_j, min(_n_each, len(_fake_j))) +
        random.sample(_real_j, min(_n_each, len(_real_j)))
    )
    print(f"Pre-capped JSON files to {len(dataset_files)} (3× sample cap) to speed up loading")
elif dataset_type in ("fakenewsnet_csv", "generic_csv") and len(dataset_files) > 20:
    dataset_files = dataset_files[:20]
    print(f"Using first 20 CSV files only")

print("\n=== Loading dataset ===")
if dataset_type == "fakenewsnet_json":
    df_raw = load_from_json(dataset_files)
elif dataset_type in ("fakenewsnet_csv", "generic_csv"):
    df_raw = load_from_csv(dataset_files)
else:
    raise RuntimeError(
        "\n\nNo usable dataset found under /kaggle/input/\n"
        "─────────────────────────────────────────────────────\n"
        "TO FIX — attach one of these Kaggle datasets:\n"
        "  Option A (recommended): Search 'FakeNewsNet' on Kaggle Datasets\n"
        "             In notebook → Add data → search 'fakenewsnet'\n"
        "             Dataset should contain gossipcop/ and politifact/ folders\n"
        "  Option B: Search 'fake news' CSV datasets (GossipCop, PolitiFact)\n"
        "             Any CSV with a title/text column + fake/real label column works\n"
        "─────────────────────────────────────────────────────"
    )

if df_raw is None or len(df_raw) < 100:
    raise RuntimeError(
        f"Dataset loaded only {len(df_raw) if df_raw is not None else 0} rows — too few.\n"
        "Check the file structure printed above and ensure the dataset is correct."
    )

print(f"\nRaw rows loaded : {len(df_raw)}")
print(f"Label dist      : {df_raw['label'].value_counts().to_dict()}")
if "img_path" in df_raw.columns:
    print(f"With image      : {df_raw['img_path'].notna().sum()}")

# ── STEP 5: Balance & oversample to MAX_SAMPLES ───────────
per_class = MAX_SAMPLES // 2   # 2,500 each
counts    = df_raw["label"].value_counts()
print(f"\nOversampling to {per_class} per class from: {counts.to_dict()}")

parts = []
for cls in [0, 1]:
    available = df_raw[df_raw["label"] == cls]
    if len(available) < 10:
        raise RuntimeError(
            f"Class {cls} has only {len(available)} samples — too few even to oversample.\n"
            "Attach a larger dataset or check label mapping above."
        )
    need_replace = len(available) < per_class
    if need_replace:
        print(f"  Class {cls}: {len(available)} real rows → oversampling to {per_class} "
              f"(repeat factor ×{per_class/len(available):.1f})")
    parts.append(available.sample(per_class, replace=need_replace, random_state=SEED))

df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Final dataset   : {len(df)} rows | {df['label'].value_counts().to_dict()}")

# ── Dataset class ─────────────────────────────────────────
_GREY = Image.new("RGB", (224, 224), (128, 128, 128))

class FakeNewsNetDataset(Dataset):
    def __init__(self, df, clip_proc, vilt_proc):
        self.df = df.reset_index(drop=True)
        self.cp = clip_proc
        self.vp = vilt_proc

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = str(row["text"])[:512]
        label = int(row["label"])

        img_path = row.get("img_path")
        try:
            if img_path and os.path.isfile(str(img_path)):
                image = Image.open(img_path).convert("RGB").resize((224, 224))
            else:
                image = _GREY.copy()
        except Exception:
            image = _GREY.copy()

        clip_in = self.cp(text=[text], images=image, return_tensors="pt",
                          padding="max_length", truncation=True, max_length=77)
        clip_in = {k: v.squeeze(0) for k, v in clip_in.items()}

        vilt_in = self.vp(text=[text], images=image, return_tensors="pt",
                          padding="max_length", truncation=True, max_length=40)
        vilt_in = {k: v.squeeze(0) for k, v in vilt_in.items()}

        return clip_in, vilt_in, label

def collate_fn(batch):
    clip_b, vilt_b, labels = zip(*batch)
    def stack(lst): return {k: torch.stack([d[k] for d in lst]) for k in lst[0]}
    return stack(clip_b), stack(vilt_b), torch.tensor(labels)

def make_loader(split_df, clip_proc, vilt_proc, shuffle):
    ds = FakeNewsNetDataset(split_df, clip_proc, vilt_proc)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True,
                      drop_last=shuffle)

# ── Model ─────────────────────────────────────────────────
class CLIPFeatureExtractor(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        d = clip_model.config.projection_dim
        self.fc = nn.Sequential(nn.Linear(d*2, 512), nn.GELU(), nn.Dropout(0.3))

    def forward(self, x):
        t = F.normalize(self.clip.text_projection(
            self.clip.text_model(input_ids=x["input_ids"],
                                 attention_mask=x["attention_mask"]).pooler_output), dim=-1)
        v = F.normalize(self.clip.visual_projection(
            self.clip.vision_model(pixel_values=x["pixel_values"]).pooler_output), dim=-1)
        return self.fc(torch.cat([t, v], dim=-1))

class ViLTFeatureExtractor(nn.Module):
    def __init__(self, vilt_model):
        super().__init__()
        self.vilt = vilt_model
        self.fc   = nn.Sequential(
            nn.Linear(vilt_model.config.hidden_size, 512), nn.GELU(), nn.Dropout(0.3))

    def forward(self, x):
        out = self.vilt(input_ids=x["input_ids"], attention_mask=x["attention_mask"],
                        token_type_ids=x.get("token_type_ids"),
                        pixel_values=x["pixel_values"], pixel_mask=x.get("pixel_mask"))
        return self.fc(out.last_hidden_state[:, 0, :])

class GateNetwork(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d*2, 256), nn.ReLU(),
                                 nn.Dropout(0.2), nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, c, v): return self.net(torch.cat([c, v], dim=-1))

class GatedFusionClassifier(nn.Module):
    def __init__(self, clip_model, vilt_model, num_classes=2, freeze=True):
        super().__init__()
        self.clip_fe = CLIPFeatureExtractor(clip_model)
        self.vilt_fe = ViLTFeatureExtractor(vilt_model)
        self.gate    = GateNetwork()
        self.head    = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, 256),
                                     nn.GELU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
        if freeze:
            for p in self.clip_fe.clip.parameters(): p.requires_grad = False
            for p in self.vilt_fe.vilt.parameters(): p.requires_grad = False

    def forward(self, ci, vi):
        cf = self.clip_fe(ci)
        vf = self.vilt_fe(vi)
        g  = self.gate(cf, vf)
        return self.head(g * cf + (1-g) * vf)

# ── Train / Eval helpers ──────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    scaler = make_scaler() if USE_AMP else None
    for ci, vi, labels in loader:
        ci = {k: v.to(DEVICE) for k, v in ci.items()}
        vi = {k: v.to(DEVICE) for k, v in vi.items()}
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        if USE_AMP:
            with make_autocast():
                logits = model(ci, vi); loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(ci, vi); loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total_loss += loss.item() * labels.size(0)
        preds_all  += logits.argmax(1).cpu().tolist()
        labels_all += labels.cpu().tolist()
    n = len(labels_all)
    return (total_loss/n,
            accuracy_score(labels_all, preds_all),
            f1_score(labels_all, preds_all, average="binary", zero_division=0))

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for ci, vi, labels in loader:
        ci = {k: v.to(DEVICE) for k, v in ci.items()}
        vi = {k: v.to(DEVICE) for k, v in vi.items()}
        labels = labels.to(DEVICE)
        if USE_AMP:
            with make_autocast():
                logits = model(ci, vi); loss = criterion(logits, labels)
        else:
            logits = model(ci, vi); loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds_all  += logits.argmax(1).cpu().tolist()
        labels_all += labels.cpu().tolist()
    n = len(labels_all)
    return (total_loss/n,
            accuracy_score(labels_all, preds_all),
            f1_score(labels_all, preds_all, average="binary", zero_division=0),
            precision_score(labels_all, preds_all, average="binary", zero_division=0),
            recall_score(labels_all,    preds_all, average="binary", zero_division=0),
            preds_all, labels_all)

# ── 5-Fold Cross-Validation on oversampled 5000-sample dataset ───────────
print("\nLoading processors…")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
vilt_processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
labels_arr   = df["label"].values
fold_results = []

print("\n" + "="*75)
print(f"  Gated Fusion CLIP+ViLT  |  FakeNewsNet 2-Way  |  {len(df)} Samples")
print(f"  {N_FOLDS}-Fold CV  |  {NUM_EPOCHS} Epochs  |  Batch {BATCH_SIZE}  |  LR {LR}")
print("="*75)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df, labels_arr)):
    print(f"\n{'─'*75}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*75}")

    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df      = df.iloc[test_idx].reset_index(drop=True)

    train_df, val_df = train_test_split(
        train_val_df, test_size=VAL_SPLIT,
        stratify=train_val_df["label"], random_state=SEED)
    print(f"  Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

    train_loader = make_loader(train_df, clip_processor, vilt_processor, shuffle=True)
    val_loader   = make_loader(val_df,   clip_processor, vilt_processor, shuffle=False)
    test_loader  = make_loader(test_df,  clip_processor, vilt_processor, shuffle=False)

    clip_base = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    vilt_base = ViltModel.from_pretrained("dandelin/vilt-b32-mlm")
    model     = GatedFusionClassifier(clip_base, vilt_base,
                                      num_classes=NUM_CLASSES,
                                      freeze=FREEZE_BACKBONES).to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"  Params: {total_p/1e6:.1f}M total | {trainable/1e6:.2f}M trainable")

    cc = np.bincount(train_df["label"].values, minlength=NUM_CLASSES)
    cw = 1.0 / (cc + 1e-6); cw = cw / cw.sum() * NUM_CLASSES
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE),
        label_smoothing=0.1)

    total_steps = len(train_loader) * NUM_EPOCHS
    optimizer   = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=1e-2)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)

    best_val_f1 = 0.0; best_state = None; no_improve = 0; history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch(
            model, train_loader, optimizer, scheduler, criterion)
        vl_loss, vl_acc, vl_f1, vl_pr, vl_rc, _, _ = eval_epoch(
            model, val_loader, criterion)
        gap = round(tr_acc - vl_acc, 4)
        history.append({"ep": epoch, "tr_acc": tr_acc,
                        "vl_acc": vl_acc, "vl_f1": vl_f1, "gap": gap})

        print(f"  Ep {epoch:02d}/{NUM_EPOCHS}  "
              f"TR={tr_acc:.4f}/F1={tr_f1:.4f}  |  "
              f"VL={vl_acc:.4f}/F1={vl_f1:.4f}  "
              f"Prec={vl_pr:.4f}  Rec={vl_rc:.4f}  Gap={gap:+.4f}")

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve  = 0
            ckpt_path   = os.path.join(CKPT_DIR, f"fold{fold+1}_best.pt")
            torch.save({"fold": fold+1, "epoch": epoch, "val_f1": vl_f1,
                        "state_dict": best_state}, ckpt_path)
            print(f"  ✓ Checkpoint (val_f1={vl_f1:.4f}) → {ckpt_path}")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PAT:
                print(f"  Early stop at epoch {epoch}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, ts_acc, ts_f1, ts_pr, ts_rc, ts_preds, ts_labels = eval_epoch(
        model, test_loader, criterion)
    final_gap = round(history[-1]["tr_acc"] - history[-1]["vl_acc"], 4)

    print(f"\n  ✅ Fold {fold+1} TEST  "
          f"Acc={ts_acc:.4f}  F1={ts_f1:.4f}  "
          f"Prec={ts_pr:.4f}  Rec={ts_rc:.4f}  Gap={final_gap:+.4f}")
    print("\n  Classification Report:")
    print(classification_report(ts_labels, ts_preds,
                                target_names=CLASS_NAMES, zero_division=0))

    fold_results.append({
        "Fold":        fold + 1,
        "Test Acc":    round(ts_acc, 4),
        "Test F1":     round(ts_f1,  4),
        "Precision":   round(ts_pr,  4),
        "Recall":      round(ts_rc,  4),
        "Best Val F1": round(best_val_f1, 4),
        "Overfit Gap": final_gap,
        "Epochs Run":  len(history),
    })

    del model, clip_base, vilt_base
    torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────
print("\n" + "="*75)
print(f"  FINAL {N_FOLDS}-FOLD RESULTS  —  Gated Fusion CLIP+ViLT  |  {len(df)} Samples")
print("="*75)

results_df = pd.DataFrame(fold_results)
mean_row = results_df.drop(columns=["Fold", "Epochs Run"]).mean().round(4).to_dict()
std_row  = results_df.drop(columns=["Fold", "Epochs Run"]).std().round(4).to_dict()
mean_row.update({"Fold": "Mean", "Epochs Run": "-"})
std_row.update( {"Fold": "Std",  "Epochs Run": "-"})
summary_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])],
                       ignore_index=True)

if HAS_TABULATE:
    print(tabulate(summary_df, headers="keys", tablefmt="double_grid",
                   showindex=False, floatfmt=".4f"))
else:
    print(summary_df.to_string(index=False))
print("="*75)

mean_acc = results_df["Test Acc"].mean()
mean_f1  = results_df["Test F1"].mean()
mean_gap = results_df["Overfit Gap"].mean()
print(f"\n  Mean Test Accuracy : {mean_acc:.4f}  ({mean_acc*100:.2f}%)")
print(f"  Mean Test F1-Score : {mean_f1:.4f}  ({mean_f1*100:.2f}%)")
print(f"  Mean Overfit Gap   : {mean_gap:+.4f}")

out_csv = "/kaggle/working/fold_results.csv"
summary_df.to_csv(out_csv, index=False)
print(f"\nResults saved: {out_csv}")

try:
    from IPython.display import display
    display(summary_df.style
        .set_caption(
            f"Gated Fusion CLIP+ViLT — FakeNewsNet 2-Way | "
            f"{N_FOLDS}-Fold CV | {len(df)} Samples (oversampled)")
        .background_gradient(
            subset=["Test Acc", "Test F1", "Precision", "Recall"], cmap="YlGn")
        .applymap(lambda v: "color:red"   if isinstance(v, float) and v >  0.05
                  else ("color:green"     if isinstance(v, float) and v <= 0.02 else ""),
                  subset=["Overfit Gap"])
        .format(precision=4))
except Exception:
    pass

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

=== Scanning /kaggle/input ===
Total files : 14
Total dirs  : 4
Attached datasets: ['datasets']

First 60 files found:
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeedNews.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeedNewsUser.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeedUser.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeedUserFeature.mat
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeedUserUser.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeed_fake_news_content.csv
  /kaggle/input/datasets/mdepak/fakenewsnet/BuzzFeed_real_news_content.csv
  /kaggle/input/datasets/mdepak/fakenewsnet/PolitiFactNews.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/PolitiFactNewsUser.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/PolitiFactUser.txt
  /kaggle/input/datasets/mdepak/fakenewsnet/PolitiFactUserFeature.mat
  /kaggle/input/datasets/mdepak/fakenewsnet/PolitiFactUserUser.txt
  /kaggle/input/datasets/mdepak

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Params: 264.2M total | 1.31M trainable
  Ep 01/10  TR=0.5564/F1=0.5649  |  VL=0.5700/F1=0.2773  Prec=0.8684  Rec=0.1650  Gap=-0.0136
  ✓ Checkpoint (val_f1=0.2773) → /kaggle/working/checkpoints/fold5_best.pt
  Ep 02/10  TR=0.6353/F1=0.6184  |  VL=0.7100/F1=0.6352  Prec=0.8559  Rec=0.5050  Gap=-0.0747
  ✓ Checkpoint (val_f1=0.6352) → /kaggle/working/checkpoints/fold5_best.pt
  Ep 03/10  TR=0.6752/F1=0.6634  |  VL=0.7025/F1=0.6972  Prec=0.7098  Rec=0.6850  Gap=-0.0273
  ✓ Checkpoint (val_f1=0.6972) → /kaggle/working/checkpoints/fold5_best.pt
  Ep 04/10  TR=0.6855/F1=0.6747  |  VL=0.6725/F1=0.6843  Prec=0.6605  Rec=0.7100  Gap=+0.0130
  Ep 05/10  TR=0.7079/F1=0.6934  |  VL=0.6900/F1=0.7033  Prec=0.6743  Rec=0.7350  Gap=+0.0179
  ✓ Checkpoint (val_f1=0.7033) → /kaggle/working/checkpoints/fold5_best.pt
  Ep 06/10  TR=0.7095/F1=0.7004  |  VL=0.6550/F1=0.6387  Prec=0.6703  Rec=0.6100  Gap=+0.0545
  Ep 07/10  TR=0.7062/F1=0.6947  |  VL=0.7125/F1=0.6779  Prec=0.7707  Rec=0.6050  Gap=-0.0063
 

,Fold,Test Acc,Test F1,Precision,Recall,Best Val F1,Overfit Gap,Epochs Run
0,1,0.6700,0.7170,0.6276,0.8360,0.7155,0.0404,6
1,2,0.6670,0.6902,0.6452,0.7420,0.6833,0.0446,8
2,3,0.6610,0.6992,0.6284,0.7880,0.6993,0.0829,9
3,4,0.6690,0.7015,0.6388,0.7780,0.6930,0.0854,7
4,5,0.6910,0.7003,0.6798,0.7220,0.7033,0.0512,8
5,Mean,0.6716,0.7016,0.6440,0.7732,0.6989,0.0609,-
6,Std,0.0114,0.0097,0.0213,0.0441,0.0120,0.0216,-
